## 介紹

本課程將涵蓋：
- 什麼是函式呼叫及其使用案例
- 如何使用 OpenAI 建立函式呼叫
- 如何將函式呼叫整合到應用程式中

## 學習目標

完成本課程後，您將了解並能：

- 使用函式呼叫的目的
- 使用 OpenAI 服務設定函式呼叫
- 為您的應用程式使用案例設計有效的函式呼叫


## 理解函式呼叫

在本課程中，我們想為我們的教育新創公司建立一個功能，讓用戶能使用聊天機器人來尋找技術課程。我們會根據用戶的技能水平、目前職位及感興趣的技術來推薦課程。

為了完成此目標，我們將使用以下組合：
 - 使用 `OpenAI` 為用戶建立聊天體驗
 - 使用 `Microsoft Learn Catalog API` 根據用戶的需求幫助尋找課程
 - 使用 `函式呼叫` 將用戶的查詢發送給函式以進行 API 請求。

現在開始，我們先來看看為什麼我們會想要使用函式呼叫：


### 為什麼要使用函式呼叫

如果你已經完成本課程中的其他課程，你大概了解使用大型語言模型 (LLMs) 的強大功能。希望你也能看到它們的一些限制。

函式呼叫是 OpenAI 服務中的一項功能，旨在解決以下挑戰：

回應格式不一致：
- 在函式呼叫之前，大型語言模型的回應是無結構且不一致的。開發者必須撰寫複雜的驗證程式碼來處理每種不同的輸出變化。

與外部數據的整合有限：
- 在此功能推出之前，很難將應用程序其他部分的數據整合進聊天上下文中。

透過標準化回應格式並啟用與外部數據的無縫整合，函式呼叫簡化了開發流程並減少了額外驗證邏輯的需求。

使用者無法獲得像「斯德哥爾摩當前天氣如何？」這樣的答案。這是因為模型的知識被限制在訓練資料的時間點。

讓我們來看下面的示例說明這個問題：

假設我們想建立一個學生資料庫，以便能推薦適合的課程給他們。以下是兩個學生的描述，它們所含的資料非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想要將此發送給大型語言模型 (LLM) 以解析數據。這之後可以用於我們的應用程式中，將其發送到 API 或儲存到資料庫。

讓我們建立兩個相同的提示，指示 LLM 我們感興趣的資訊：


我們想將這個傳送給大型語言模型（LLM），以解析對我們產品重要的部分。因此，我們可以建立兩個相同的提示來指示LLM：


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


在建立這兩個提示後，我們將使用 `client.responses.create` 將它們發送到 LLM。 我們將提示存儲在 `input` 變數中，並將角色指定為 `user`。這是為了模擬用戶向聊天機器人寫入訊息。 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-5-mini"


: 

現在我們可以同時將兩個請求發送給 LLM，並檢視我們收到的回應。 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



儘管提示詞相同且描述相似，我們仍可能得到不同格式的 `Grades` 屬性。 

如果你多次執行上面的單元格，格式可能是 `3.7` 或 `3.7 GPA`。 

這是因為大型語言模型接收以書面提示形式呈現的非結構化資料，並回傳同樣是非結構化資料。我們需要有結構化的格式，才能知道在儲存或使用這些資料時應該期待什麼內容。

透過使用函式呼叫，我們可以確保收到的是結構化資料。在使用函式呼叫時，LLM 其實並不會真正呼叫或執行任何函式，而是我們為 LLM 建立一個結構，讓它依照該結構回應。我們再根據這些結構化的回應，知道要在應用程式中執行哪個函式。  
 


![函式呼叫流程圖](../../../../translated_images/zh-TW/Function-Flow.083875364af4f4bb.webp)


接著我們可以將函式回傳的結果拿去送回給 LLM。LLM 接著會用自然語言來回應，以回答使用者的查詢。


### 使用函式呼叫的應用案例 

<strong>呼叫外部工具</strong> 
聊天機器人在提供使用者問題答案方面表現優異。透過函式呼叫，聊天機器人可以利用使用者的訊息來完成特定任務。例如，學生可以要求聊天機器人「寄信給我的講師，說我需要更多這科目的協助」。這時可以呼叫函式 `send_email(to: string, body: string)`


**建立 API 或資料庫查詢**
使用者可以用自然語言查詢資訊，並轉換成格式化的查詢或 API 請求。例如，一位老師若請求「誰是完成最後一次作業的學生」，就可以呼叫名為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函式


<strong>建立結構化資料</strong>
使用者可以將一段文字或 CSV 檔案交給大型語言模型以擷取重要資訊。例如，學生可以將維基百科中關於和平協議的文章轉換成 AI 單字卡。這可以透過名為 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 的函式來完成


## 2. 建立您的第一個函式呼叫 

建立函式呼叫的過程包括三個主要步驟： 
1. 使用您的函式清單和使用者訊息呼叫 Chat Completions API 
2. 讀取模型的回應以執行一個動作，即執行函式或 API 呼叫 
3. 使用來自函式的回應，再次呼叫 Chat Completions API，利用該資訊來建立給使用者的回應。 


![函式呼叫流程](../../../../translated_images/zh-TW/LLM-Flow.3285ed8caf4796d7.webp)


### 函式呼叫的元素 

#### 使用者輸入 

第一步是創建一則使用者訊息。這可以透過取得文字輸入的值動態指派，或者你也可以在此直接指派一個值。如果你是第一次使用 Chat Completions API，我們需要定義訊息的 `role` 與 `content`。 

`role` 可設定為 `system`（制定規則）、`assistant`（模型）或 `user`（終端使用者）。針對函式呼叫，我們會將其指定為 `user`，並給出一個範例問題。 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 創建函數。

接下來我們將定義一個函數及其參數。我們這裡只使用一個函數，名為 `search_courses`，但你也可以創建多個函數。

<strong>重要</strong>：函數會被包含在系統訊息中傳送給 LLM，並且會計入你可用的總令牌數量。


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

函數定義結構有多個層級，每個層級都有其屬性。以下是巢狀結構的解析：

**頂層函數屬性：**

`name` - 我們希望被呼叫的函數名稱。

`description` - 這是函數運作方式的說明。此處重要的是要具體且清楚。

`parameters` - 你希望模型在回應中產生的值和格式清單。

**參數物件屬性：**

`type` - 參數物件的資料型態（通常是 "object"）

`properties` - 模型將用於回應的具體值清單

**個別參數屬性：**

`name` - 由屬性鍵隱含定義 (例如 "role", "product", "level")

`type` - 此特定參數的資料型態（例如 "string", "number", "boolean"）

`description` - 此特定參數的描述

**選用屬性：**

`required` - 以陣列列出完成函數呼叫所需的參數


### 呼叫函式 
定義函式之後，我們現在需要在呼叫 Chat Completion API 時包含該函式。我們可以透過在請求中添加 `functions` 來做到這點。在此範例中是 `functions=functions`。

也可以設定 `function_call` 為 `auto`。這表示我們將讓 LLM 根據使用者的訊息自行決定要呼叫哪個函式，而不是由我們自己指定。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


現在讓我們來看看回應並了解它是如何格式化的：

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以看到該函式名稱被呼叫，並且根據使用者的訊息，LLM 能夠找到符合函式參數的資料。


## 3.將函式呼叫整合到應用程式中。


在我們測試完 LLM 格式化回應後，現在可以將其整合到應用程式中。

### 流程管理

要將此整合到我們的應用程式中，讓我們採取以下步驟：

首先，呼叫 OpenAI 服務並將訊息儲存在名為 `response_message` 的變數中。


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函式，用來呼叫 Microsoft Learn API 取得課程清單： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為最佳實踐，我們接著會檢查模型是否想要呼叫一個函式。之後，我們會建立一個現有的函式並將其對應到正在被呼叫的函式。
接著，我們會取得該函式的參數，並將它們對應到來自 LLM 的參數。

最後，我們會附加函式呼叫訊息以及由 `search_courses` 訊息回傳的值。這會提供 LLM 回應使用者所需的所有資訊，
使用自然語言來回覆使用者。


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們將把更新後的訊息傳送給 LLM，以便我們能收到自然語言回應，而非 API JSON 格式的回應。


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 程式碼挑戰 

太棒了！要繼續學習 OpenAI 函數呼叫，您可以建置：https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 更多函數參數可能幫助學習者找到更多課程。您可以在這裡找到可用的 API 參數： 
 - 建立另一個函數呼叫，從學習者那裡獲取更多資訊，例如他們的母語 
 - 當函數呼叫和/或 API 呼叫沒有回傳合適課程時，建立錯誤處理 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
